In [ ]:
# imports the necessary requirements
from crewai import Crew, Agent, Task
from langchain_openai import ChatOpenAI
import pm4py
import duckdb
import os
from crewai_tools import tool
import pickle

In [ ]:
# sets the API key (always needed!)
os.environ["OPENAI_API_KEY"] = open("api_key.txt", "r").read()

In [ ]:
# OpenAI's GPT-4o Mini model 
llm = ChatOpenAI(model="gpt-4o-mini", base_url="https://api.openai.com/v1")

In [ ]:
question = open('questions\cat02_10_root_cause_1 + preprocessing.txt', 'r', encoding='utf-8').read()

In [ ]:
q02_10_analyst = Agent(role="process_analyst", goal="Examine logs to determine the root causeof delays", backstory="Very detailed analyst.", llm=llm)
q02_10_especialist = Agent(role="process_reviewer", goal="Review the analyst's report and identify areas for improvement", backstory="Senior Especialist", llm=llm)
q02_10_manager = Agent(role="process_manager", goal="Adjust the analyst's reply to address the especialist insights", backstory="Process Manager", llm=llm)

In [ ]:
analise = Task(description=question, expected_output="A proper awnser to the question", agent=q02_10_analyst)
review = Task(description="Please evaluate with the most strictness. Be hypercritical of any inaccuracies, unclarities, or logical flaws. Even minor issues should result in a significantly lower score. Only award a very high score if the answer is nearly flawless.", expected_output="The chain of thought related to the insight", agent=q02_10_especialist)
final_answer = Task(description="Adjust the response with the evaluation of the specialist. If the specialist evaluation is not satisfactory, make necessary corrections to improve the response. Ensure that the final answer is clear, accurate, and meets the highest standards of quality.", expected_output="The final answer to the question",  agent=q02_10_manager,)

In [ ]:
crew = Crew(agents=[q02_10_analyst, q02_10_especialist, q02_10_manager], tasks=[analise, review, final_answer], verbose=True)

In [ ]:
crew.kickoff()